# Visual Search System - Advanced Topics & References

This notebook covers advanced topics and extensions for building production-grade visual search systems. We'll explore content moderation, handling position bias, multi-modal search, and other advanced techniques.

## Learning Objectives
- Implement content moderation for search results
- Handle position bias in click data
- Leverage image metadata for improved search
- Explore multi-modal search (text + image)
- Understand active learning for efficient annotation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Dict, Tuple

## 1. Content Moderation

Visual search results may contain inappropriate or harmful content. Content moderation filters ensure safe results.

### 1.1 Types of Inappropriate Content
- Adult/NSFW content
- Violence and gore
- Hate symbols
- Spam/scam images
- Copyright violations

In [ ]:
class ContentModerationService:
    """
    Filters inappropriate content from search results.
    """
    def __init__(self, config: Dict = None):
        self.config = config or {
            'nsfw_threshold': 0.9,
            'violence_threshold': 0.85,
            'spam_threshold': 0.8
        }
        # In production, would load actual moderation models
        self.nsfw_model = None
        self.violence_model = None
        self.spam_model = None
    
    def classify_image(self, image_embedding: np.ndarray) -> Dict[str, float]:
        """
        Classify image for different types of inappropriate content.
        
        Returns:
            Dict with probability scores for each category
        """
        # Simulate classification scores
        return {
            'nsfw_score': np.random.random() * 0.3,  # Usually low
            'violence_score': np.random.random() * 0.2,
            'spam_score': np.random.random() * 0.4
        }
    
    def should_filter(self, scores: Dict[str, float]) -> Tuple[bool, str]:
        """
        Determine if image should be filtered.
        
        Returns:
            (should_filter, reason)
        """
        if scores['nsfw_score'] > self.config['nsfw_threshold']:
            return True, 'nsfw'
        if scores['violence_score'] > self.config['violence_threshold']:
            return True, 'violence'
        if scores['spam_score'] > self.config['spam_threshold']:
            return True, 'spam'
        return False, None
    
    def filter_results(self, results: List[Tuple[str, float]]) -> List[Tuple[str, float]]:
        """
        Filter search results for inappropriate content.
        """
        filtered = []
        removed_count = 0
        
        for img_id, score in results:
            # Simulate getting embedding (in production, would use cached embeddings)
            embedding = np.random.randn(128)
            mod_scores = self.classify_image(embedding)
            
            should_filter, reason = self.should_filter(mod_scores)
            if not should_filter:
                filtered.append((img_id, score))
            else:
                removed_count += 1
        
        print(f"Removed {removed_count} images due to content policy")
        return filtered

# Example
moderation = ContentModerationService()
sample_results = [(f"img_{i}", 0.9 - i*0.05) for i in range(20)]
filtered_results = moderation.filter_results(sample_results)
print(f"Original: {len(sample_results)}, After filtering: {len(filtered_results)}")

### 1.2 Moderation Pipeline Integration

Content moderation can be applied at different stages:

1. **Indexing Time (Proactive)**: Check all images before adding to index
2. **Query Time (Reactive)**: Filter results during search
3. **Hybrid**: Pre-filter most content, do additional checks at query time

In [ ]:
def visualize_moderation_pipeline():
    """Visualize content moderation integration"""
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Define pipeline stages
    stages = [
        {'name': 'Image\nUpload', 'x': 0.1, 'color': 'lightblue'},
        {'name': 'Pre-\nModeration', 'x': 0.25, 'color': 'lightcoral'},
        {'name': 'Embedding\nGeneration', 'x': 0.4, 'color': 'lightblue'},
        {'name': 'Index\nStorage', 'x': 0.55, 'color': 'lightgreen'},
        {'name': 'Search\nQuery', 'x': 0.7, 'color': 'lightyellow'},
        {'name': 'Post-\nModeration', 'x': 0.85, 'color': 'lightcoral'},
    ]
    
    for stage in stages:
        rect = plt.Rectangle((stage['x']-0.06, 0.3), 0.12, 0.4,
                            facecolor=stage['color'], edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(stage['x'], 0.5, stage['name'], ha='center', va='center',
               fontsize=10, fontweight='bold')
    
    # Draw arrows
    for i in range(len(stages)-1):
        ax.annotate('', xy=(stages[i+1]['x']-0.06, 0.5),
                   xytext=(stages[i]['x']+0.06, 0.5),
                   arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Labels
    ax.text(0.25, 0.15, 'Block 5% of uploads', ha='center', fontsize=9, style='italic')
    ax.text(0.85, 0.15, 'Additional 0.1% filtered', ha='center', fontsize=9, style='italic')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title('Content Moderation in Visual Search Pipeline', fontsize=14, fontweight='bold')
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='lightcoral', label='Moderation Stage'),
        Patch(facecolor='lightblue', label='Processing Stage'),
        Patch(facecolor='lightgreen', label='Storage Stage'),
        Patch(facecolor='lightyellow', label='Query Stage')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()

visualize_moderation_pipeline()

## 2. Handling Position Bias

Users tend to click on results at higher positions regardless of relevance. This creates **position bias** in click data.

### 2.1 Position Bias in Click Data

In [ ]:
def visualize_position_bias():
    """Visualize position bias in search results"""
    positions = np.arange(1, 21)
    
    # Simulated CTR by position (typical decay pattern)
    ctr = 0.3 * np.exp(-0.2 * (positions - 1)) + 0.02
    
    # True relevance (assume uniform for illustration)
    true_relevance = np.ones_like(positions) * 0.15
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.bar(positions - 0.2, ctr * 100, width=0.4, label='Observed CTR', color='steelblue', alpha=0.8)
    ax.bar(positions + 0.2, true_relevance * 100, width=0.4, label='True Relevance', color='orange', alpha=0.8)
    
    ax.set_xlabel('Position', fontsize=12)
    ax.set_ylabel('Rate (%)', fontsize=12)
    ax.set_title('Position Bias: CTR vs True Relevance', fontsize=14)
    ax.set_xticks(positions)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add annotation
    ax.annotate('Position 1 gets 5x more clicks\nthan position 10', 
               xy=(1, ctr[0]*100), xytext=(5, 25),
               arrowprops=dict(arrowstyle='->', color='red'),
               fontsize=10, color='red')
    
    plt.tight_layout()
    plt.show()

visualize_position_bias()

### 2.2 Debiasing Techniques

In [ ]:
class PositionBiasHandler:
    """
    Handle position bias in click data.
    """
    def __init__(self):
        # Propensity scores (probability of examination at each position)
        # Learned from randomization experiments or assumed model
        self.propensity_scores = None
    
    def estimate_propensity(self, click_data: pd.DataFrame) -> np.ndarray:
        """
        Estimate propensity scores from click data.
        
        Method: Use result randomization experiment or position-based model.
        """
        # Simplified: exponential decay model
        max_position = 20
        self.propensity_scores = np.exp(-0.2 * np.arange(max_position))
        return self.propensity_scores
    
    def inverse_propensity_weighting(self, clicks: np.ndarray, 
                                      positions: np.ndarray) -> np.ndarray:
        """
        Apply Inverse Propensity Weighting (IPW) to clicks.
        
        Debiased_signal = click / propensity(position)
        """
        propensity = self.propensity_scores[positions]
        weighted_clicks = clicks / propensity
        return weighted_clicks

# Example
handler = PositionBiasHandler()
propensity = handler.estimate_propensity(None)

print("Propensity Scores by Position:")
for i in range(10):
    print(f"  Position {i+1}: {propensity[i]:.4f}")

print("\nIPW Example:")
print("  Click at position 1: 1 / 1.000 = 1.000 (no boost)")
print(f"  Click at position 5: 1 / {propensity[4]:.3f} = {1/propensity[4]:.3f} (boosted)")
print(f"  Click at position 10: 1 / {propensity[9]:.3f} = {1/propensity[9]:.3f} (highly boosted)")

### 2.3 Other Debiasing Methods

| Method | Description | Pros | Cons |
|--------|-------------|------|------|
| **IPW** | Weight by inverse propensity | Simple, unbiased | High variance |
| **Position Model** | Model P(click) = P(examine) × P(relevant) | Interpretable | Needs assumptions |
| **Randomization** | Show results in random order | Gold standard | Hurts user experience |
| **Interleaving** | Mix models, compare clicks | Online evaluation | Complex to implement |

## 3. Using Image Metadata

Combining image embeddings with metadata can improve search quality.

### 3.1 Available Metadata
- User-provided tags
- Auto-generated captions
- Object detection labels
- Color histograms
- Image quality scores

In [ ]:
class MetadataEnhancedSearch:
    """
    Enhance visual search with metadata.
    """
    def __init__(self, visual_weight: float = 0.7, metadata_weight: float = 0.3):
        self.visual_weight = visual_weight
        self.metadata_weight = metadata_weight
    
    def compute_metadata_similarity(self, query_metadata: Dict, 
                                    candidate_metadata: Dict) -> float:
        """
        Compute metadata-based similarity.
        """
        score = 0.0
        
        # Tag overlap (Jaccard similarity)
        if 'tags' in query_metadata and 'tags' in candidate_metadata:
            q_tags = set(query_metadata['tags'])
            c_tags = set(candidate_metadata['tags'])
            if q_tags or c_tags:
                jaccard = len(q_tags & c_tags) / len(q_tags | c_tags)
                score += 0.5 * jaccard
        
        # Color similarity
        if 'dominant_color' in query_metadata and 'dominant_color' in candidate_metadata:
            if query_metadata['dominant_color'] == candidate_metadata['dominant_color']:
                score += 0.3
        
        # Category match
        if 'category' in query_metadata and 'category' in candidate_metadata:
            if query_metadata['category'] == candidate_metadata['category']:
                score += 0.2
        
        return score
    
    def combined_ranking(self, visual_scores: List[float], 
                         metadata_scores: List[float]) -> List[float]:
        """
        Combine visual and metadata scores.
        """
        combined = []
        for v, m in zip(visual_scores, metadata_scores):
            score = self.visual_weight * v + self.metadata_weight * m
            combined.append(score)
        return combined

# Example
enhanced_search = MetadataEnhancedSearch()

query_meta = {
    'tags': ['sunset', 'beach', 'ocean'],
    'dominant_color': 'orange',
    'category': 'nature'
}

candidate_meta = {
    'tags': ['sunset', 'mountains', 'landscape'],
    'dominant_color': 'orange',
    'category': 'nature'
}

meta_score = enhanced_search.compute_metadata_similarity(query_meta, candidate_meta)
print(f"Metadata similarity score: {meta_score:.4f}")

# Combined ranking example
visual_scores = [0.9, 0.85, 0.8, 0.75]
metadata_scores = [0.3, 0.8, 0.5, 0.9]
combined = enhanced_search.combined_ranking(visual_scores, metadata_scores)

print("\nCombined Ranking:")
for i, (v, m, c) in enumerate(zip(visual_scores, metadata_scores, combined)):
    print(f"  Result {i+1}: Visual={v:.2f}, Metadata={m:.2f}, Combined={c:.2f}")

## 4. Smart Cropping with Object Detection

When users upload images, they may want to search for specific objects within the image.

In [ ]:
class SmartCropping:
    """
    Automatically detect and crop objects for visual search.
    """
    def __init__(self):
        # In production, would load object detection model (YOLO, Faster R-CNN, etc.)
        self.detector = None
    
    def detect_objects(self, image: np.ndarray) -> List[Dict]:
        """
        Detect objects in image.
        
        Returns:
            List of detected objects with bounding boxes
        """
        # Simulate detection results
        detections = [
            {'class': 'dress', 'confidence': 0.95, 'bbox': (100, 50, 400, 500)},
            {'class': 'bag', 'confidence': 0.87, 'bbox': (420, 300, 550, 480)},
            {'class': 'shoes', 'confidence': 0.82, 'bbox': (150, 520, 350, 620)},
        ]
        return detections
    
    def suggest_crops(self, image: np.ndarray) -> List[Dict]:
        """
        Suggest crop regions for visual search.
        """
        detections = self.detect_objects(image)
        suggestions = []
        
        for det in detections:
            suggestions.append({
                'object_type': det['class'],
                'confidence': det['confidence'],
                'crop_region': det['bbox'],
                'searchable': True
            })
        
        return suggestions

# Example
smart_crop = SmartCropping()
sample_image = np.zeros((640, 640, 3))
suggestions = smart_crop.suggest_crops(sample_image)

print("Smart Crop Suggestions:")
for i, sug in enumerate(suggestions):
    print(f"\n{i+1}. {sug['object_type'].capitalize()}")
    print(f"   Confidence: {sug['confidence']:.0%}")
    print(f"   Crop region: {sug['crop_region']}")

## 5. Multi-Modal Search (Text + Image)

Modern visual search systems support combined text and image queries.

### 5.1 CLIP-style Multi-Modal Embeddings

In [ ]:
class MultiModalSearch:
    """
    Multi-modal search supporting text and image queries.
    Uses CLIP-style shared embedding space.
    """
    def __init__(self, embedding_dim: int = 512):
        self.embedding_dim = embedding_dim
        # In production: load CLIP or similar model
        self.image_encoder = None
        self.text_encoder = None
    
    def encode_image(self, image: np.ndarray) -> np.ndarray:
        """Encode image to shared embedding space."""
        # Simulate encoding
        embedding = np.random.randn(self.embedding_dim)
        return embedding / np.linalg.norm(embedding)
    
    def encode_text(self, text: str) -> np.ndarray:
        """Encode text to shared embedding space."""
        # Simulate encoding
        embedding = np.random.randn(self.embedding_dim)
        return embedding / np.linalg.norm(embedding)
    
    def combined_query(self, image: np.ndarray, text: str,
                       image_weight: float = 0.7) -> np.ndarray:
        """
        Create combined query embedding.
        
        Example use case:
        - User uploads a red dress image
        - User adds text "in blue color"
        - System finds blue dresses similar to the uploaded one
        """
        image_emb = self.encode_image(image)
        text_emb = self.encode_text(text)
        
        text_weight = 1 - image_weight
        combined = image_weight * image_emb + text_weight * text_emb
        return combined / np.linalg.norm(combined)

# Example
multimodal = MultiModalSearch()

# Text-only search
text_query = "blue summer dress"
text_embedding = multimodal.encode_text(text_query)
print(f"Text query: '{text_query}'")
print(f"Text embedding shape: {text_embedding.shape}")

# Combined image + text search
sample_image = np.zeros((224, 224, 3))
modification = "with floral pattern"
combined_embedding = multimodal.combined_query(sample_image, modification)
print(f"\nCombined query: [Image] + '{modification}'")
print(f"Combined embedding shape: {combined_embedding.shape}")

In [ ]:
def visualize_multimodal_embedding_space():
    """Visualize multi-modal embedding space"""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    np.random.seed(42)
    
    # Generate sample embeddings for different categories
    categories = [
        {'name': 'Red Dress (images)', 'center': (0.3, 0.7), 'color': 'red', 'marker': 'o'},
        {'name': 'Blue Dress (images)', 'center': (-0.3, 0.7), 'color': 'blue', 'marker': 'o'},
        {'name': '"red dress" (text)', 'center': (0.25, 0.65), 'color': 'red', 'marker': '^'},
        {'name': '"blue dress" (text)', 'center': (-0.25, 0.65), 'color': 'blue', 'marker': '^'},
        {'name': 'Cars (images)', 'center': (0.5, -0.5), 'color': 'green', 'marker': 'o'},
        {'name': '"car" (text)', 'center': (0.45, -0.45), 'color': 'green', 'marker': '^'},
    ]
    
    for cat in categories:
        points = np.array(cat['center']) + np.random.randn(5, 2) * 0.08
        ax.scatter(points[:, 0], points[:, 1], c=cat['color'], marker=cat['marker'],
                  s=100, alpha=0.7, label=cat['name'])
    
    ax.set_xlabel('Embedding Dimension 1', fontsize=12)
    ax.set_ylabel('Embedding Dimension 2', fontsize=12)
    ax.set_title('CLIP-style Multi-Modal Embedding Space\n(Images and Text in Same Space)', fontsize=14)
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_multimodal_embedding_space()

## 6. Graph Neural Networks for Better Representations

GNNs can leverage the relationship between images (e.g., co-occurrence, user interactions) to improve embeddings.

In [ ]:
def explain_gnn_for_visual_search():
    """Explain GNN application in visual search"""
    explanation = """
    GRAPH NEURAL NETWORKS FOR VISUAL SEARCH
    ═════════════════════════════════════════
    
    GRAPH CONSTRUCTION:
    ───────────────────
    • Nodes: Images in the catalog
    • Edges: Connections based on:
      - Visual similarity (k-NN in embedding space)
      - Co-clicks (users clicked both images)
      - Co-saves (users saved both images)
      - Same board/collection
    
    GNN MESSAGE PASSING:
    ────────────────────
    1. Start with CNN embeddings for each image
    2. Aggregate information from neighbor nodes
    3. Update node embeddings based on neighborhood
    4. Repeat for K layers
    
    BENEFITS:
    ─────────
    • Captures semantic relationships beyond visual similarity
    • Leverages collective user behavior
    • Handles cold-start for new images with few interactions
    • Improves recall for "style" and "aesthetic" queries
    
    ARCHITECTURE:
    ─────────────
    Image → CNN → Base Embedding → GNN Layers → Enhanced Embedding
                                      ↑
                              Graph Structure
    
    EXAMPLE:
    ────────
    A vintage dress might be visually different from modern dresses,
    but GNN can learn they're related through user co-interaction patterns.
    """
    print(explanation)

explain_gnn_for_visual_search()

## 7. Active Learning for Efficient Annotation

Instead of randomly labeling images, active learning selects the most informative samples.

In [ ]:
class ActiveLearningSelector:
    """
    Select most informative samples for human labeling.
    """
    def __init__(self, strategy: str = 'uncertainty'):
        self.strategy = strategy
    
    def uncertainty_sampling(self, predictions: np.ndarray) -> np.ndarray:
        """
        Select samples where model is most uncertain.
        For classification: highest entropy or closest to 0.5
        For search: queries with low confidence matches
        """
        # Entropy-based uncertainty
        entropy = -np.sum(predictions * np.log(predictions + 1e-10), axis=1)
        return np.argsort(entropy)[::-1]  # Most uncertain first
    
    def diversity_sampling(self, embeddings: np.ndarray, 
                          n_samples: int) -> np.ndarray:
        """
        Select diverse samples to cover embedding space.
        Uses k-medoids or similar clustering.
        """
        # Simplified: random sampling (in practice, use clustering)
        indices = np.random.choice(len(embeddings), n_samples, replace=False)
        return indices
    
    def select_samples(self, pool_embeddings: np.ndarray,
                       pool_predictions: np.ndarray,
                       n_samples: int) -> np.ndarray:
        """
        Select samples for labeling using specified strategy.
        """
        if self.strategy == 'uncertainty':
            ranked = self.uncertainty_sampling(pool_predictions)
            return ranked[:n_samples]
        elif self.strategy == 'diversity':
            return self.diversity_sampling(pool_embeddings, n_samples)
        elif self.strategy == 'hybrid':
            # Combine uncertainty and diversity
            uncertain = self.uncertainty_sampling(pool_predictions)[:n_samples*2]
            diverse = self.diversity_sampling(pool_embeddings[uncertain], n_samples)
            return uncertain[diverse]
        else:
            # Random baseline
            return np.random.choice(len(pool_embeddings), n_samples, replace=False)

# Example
selector = ActiveLearningSelector(strategy='uncertainty')

# Simulate pool of unlabeled data
np.random.seed(42)
pool_size = 1000
pool_predictions = np.random.dirichlet([1, 1, 1, 1], pool_size)  # 4-class
pool_embeddings = np.random.randn(pool_size, 128)

# Select top 50 samples for labeling
selected = selector.select_samples(pool_embeddings, pool_predictions, n_samples=50)

print("Active Learning Sample Selection:")
print(f"  Pool size: {pool_size}")
print(f"  Samples selected: {len(selected)}")
print(f"  Selection strategy: {selector.strategy}")
print(f"\nSelected sample indices (first 10): {selected[:10]}")

In [ ]:
def visualize_active_learning():
    """Visualize active learning efficiency"""
    iterations = np.arange(1, 11)
    
    # Simulated learning curves
    random_sampling = 0.5 + 0.4 * (1 - np.exp(-0.3 * iterations))
    active_learning = 0.5 + 0.45 * (1 - np.exp(-0.5 * iterations))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(iterations * 100, random_sampling, 'b-o', linewidth=2, markersize=8,
           label='Random Sampling')
    ax.plot(iterations * 100, active_learning, 'r-s', linewidth=2, markersize=8,
           label='Active Learning')
    
    ax.set_xlabel('Number of Labeled Samples', fontsize=12)
    ax.set_ylabel('Model Performance (nDCG)', fontsize=12)
    ax.set_title('Active Learning vs Random Sampling Efficiency', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.4, 1.0)
    
    # Annotation
    ax.annotate('Active learning reaches\n90% performance with\n50% fewer samples',
               xy=(500, 0.85), xytext=(600, 0.7),
               arrowprops=dict(arrowstyle='->', color='black'),
               fontsize=10,
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

visualize_active_learning()

## 8. Human-in-the-Loop ML

Combining automated ML with human judgment for continuous improvement.

In [ ]:
def human_in_the_loop_workflow():
    """Human-in-the-loop workflow for visual search"""
    workflow = """
    HUMAN-IN-THE-LOOP ML FOR VISUAL SEARCH
    ═══════════════════════════════════════
    
    ┌─────────────────────────────────────────────────────────┐
    │                    CONTINUOUS LOOP                       │
    └─────────────────────────────────────────────────────────┘
    
    1. MODEL INFERENCE
       │
       ├── Search results generated by ML model
       └── Confidence scores for each result
              │
              ▼
    2. AUTOMATED QUALITY CHECK
       │
       ├── Flag low-confidence results
       ├── Detect potential content policy violations
       └── Identify edge cases
              │
              ▼
    3. HUMAN REVIEW QUEUE
       │
       ├── Prioritized by impact & uncertainty
       ├── Human reviewers label/correct results
       └── Feedback collected
              │
              ▼
    4. DATA COLLECTION
       │
       ├── Add human judgments to training data
       ├── Log correction patterns
       └── Update content policy rules
              │
              ▼
    5. MODEL RETRAINING
       │
       ├── Periodic retraining with new data
       ├── Focus on corrected examples
       └── A/B test new model
              │
              └──────────► Back to Step 1
    
    KEY METRICS:
    ────────────
    • Human review rate: 0.1% of queries
    • Model agreement with humans: 95%+
    • Time to incorporate feedback: < 1 week
    • Continuous improvement: +2% nDCG/quarter
    """
    print(workflow)

human_in_the_loop_workflow()

## 9. References and Further Reading

### 9.1 Foundational Papers

In [ ]:
def display_references():
    """Key references for visual search systems"""
    references = {
        'Representation Learning': [
            'SimCLR: A Simple Framework for Contrastive Learning (Chen et al., 2020)',
            'MoCo: Momentum Contrast for Unsupervised Visual Representation Learning (He et al., 2020)',
            'CLIP: Learning Transferable Visual Models From Natural Language Supervision (Radford et al., 2021)',
            'DINO: Emerging Properties in Self-Supervised Vision Transformers (Caron et al., 2021)',
        ],
        'Neural Network Architectures': [
            'Deep Residual Learning for Image Recognition (He et al., 2016)',
            'An Image is Worth 16x16 Words: Transformers for Image Recognition (Dosovitskiy et al., 2020)',
            'EfficientNet: Rethinking Model Scaling (Tan & Le, 2019)',
        ],
        'Approximate Nearest Neighbor': [
            'Efficient and Robust Approximate Nearest Neighbor Search (Faiss, Johnson et al., 2019)',
            'Accelerating Large-Scale Inference with Anisotropic Vector Quantization (ScaNN, Guo et al., 2020)',
            'Hierarchical Navigable Small World Graphs (HNSW, Malkov & Yashunin, 2018)',
        ],
        'Ranking & Evaluation': [
            'Learning to Rank for Information Retrieval (Liu, 2009)',
            'Unbiased Learning to Rank with Unbiased Propensity Estimation (Joachims et al., 2017)',
            'Position Bias Estimation for Unbiased Learning to Rank (Agarwal et al., 2019)',
        ],
        'Industry Systems': [
            'Visual Search at Pinterest (Jing et al., 2015)',
            'Visual Search at eBay (Yang et al., 2017)',
            'Billion-scale Similarity Search with GPUs (Facebook, Johnson et al., 2017)',
            'Embedding-based Retrieval in Facebook Search (Huang et al., 2020)',
        ],
        'Multi-Modal Learning': [
            'ALIGN: Scaling Up Visual and Vision-Language Representation (Jia et al., 2021)',
            'BLIP: Bootstrapping Language-Image Pre-training (Li et al., 2022)',
            'Flamingo: a Visual Language Model (Alayrac et al., 2022)',
        ],
        'Graph Neural Networks': [
            'Graph Attention Networks (Veličković et al., 2017)',
            'PinSage: Graph Convolutional Neural Networks for Web-Scale Recommender Systems (Ying et al., 2018)',
        ]
    }
    
    print("KEY REFERENCES FOR VISUAL SEARCH SYSTEMS")
    print("=" * 70)
    
    for category, papers in references.items():
        print(f"\n{category}:")
        print("-" * 50)
        for paper in papers:
            print(f"  • {paper}")

display_references()

## 10. Summary

### Key Takeaways

In [ ]:
def advanced_topics_summary():
    summary = """
    ╔══════════════════════════════════════════════════════════════╗
    ║       VISUAL SEARCH ADVANCED TOPICS SUMMARY                  ║
    ╠══════════════════════════════════════════════════════════════╣
    ║                                                              ║
    ║   CONTENT MODERATION:                                        ║
    ║   ├── Pre-moderation at indexing time                       ║
    ║   ├── Post-moderation at query time                         ║
    ║   └── Multi-stage filtering for safety                      ║
    ║                                                              ║
    ║   POSITION BIAS:                                             ║
    ║   ├── Inverse Propensity Weighting (IPW)                    ║
    ║   ├── Position-based click models                           ║
    ║   └── Randomization experiments                             ║
    ║                                                              ║
    ║   METADATA & FEATURES:                                       ║
    ║   ├── Tags, captions, object detection                      ║
    ║   ├── Combined visual + metadata scoring                    ║
    ║   └── Smart cropping with object detection                  ║
    ║                                                              ║
    ║   MULTI-MODAL SEARCH:                                        ║
    ║   ├── CLIP-style shared embedding space                     ║
    ║   ├── Image + text query composition                        ║
    ║   └── Cross-modal retrieval                                 ║
    ║                                                              ║
    ║   ADVANCED TECHNIQUES:                                       ║
    ║   ├── GNNs for relationship-aware embeddings               ║
    ║   ├── Active learning for efficient labeling               ║
    ║   └── Human-in-the-loop continuous improvement             ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    """
    print(summary)

advanced_topics_summary()

## Module 2 Complete Summary

This module covered a complete Visual Search System case study, demonstrating how to design and build a production-grade visual search system similar to Pinterest's visual search.

### Notebooks in this Module:

1. **Requirements & Problem Framing**: Clarifying requirements, defining ML objectives, framing as representation learning

2. **Data & Feature Engineering**: Data sources, image preprocessing, interaction data handling

3. **Model Development**: CNN vs ViT architectures, contrastive training (SimCLR, MoCo)

4. **Evaluation Metrics**: nDCG, Precision@k, Recall@k, mAP, online metrics

5. **Serving Architecture**: Prediction pipeline, indexing pipeline, ANN algorithms

6. **Advanced Topics**: Content moderation, position bias, multi-modal search, GNNs, active learning